# 01. Dataset Exploration: Zomato Bangalore Restaurants
**Objective:** Inspect raw dataset schema, identify data quality anomalies, missing values, and duplicate records before applying Apache Spark distributed processing.

## 1. Project Introduction & Environmental Setup
We import standard exploratory libraries and verify dataset file presence.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Configurable relative dataset path
RAW_DATA_PATH = Path('../data/raw/zomato.csv')
print(f'Dataset Path exists: {RAW_DATA_PATH.exists()}')
if RAW_DATA_PATH.exists():
    print(f'File Size: {RAW_DATA_PATH.stat().st_size / (1024 * 1024):.2f} MB')

## 2. Dataset Ingestion and Schema Overview
We load a sample of the dataset using Pandas for lightweight schema inspection.

In [ ]:
# Load initial sample of raw data
raw_sample = pd.read_csv(RAW_DATA_PATH, nrows=1000)
print('Dataset Dimensions (sample preview):', raw_sample.shape)
print('\nColumn Names and Data Types:')
print(raw_sample.dtypes)
raw_sample.head(3)

## 3. Full Dataset Dimensions & Record Count
Let us inspect the full dataset record count.

In [ ]:
# Compute total record count across full raw dataset
full_df = pd.read_csv(RAW_DATA_PATH, usecols=['name', 'location', 'rate', 'votes', 'approx_cost(for two people)'])
print(f'Total Raw Records: {len(full_df):,}')
print(f'Total Attributes: 17')

## 4. Missing Value Analysis
Analyzing missingness patterns across critical business attributes.

In [ ]:
missing_stats = pd.DataFrame({
    'Null_Count': full_df.isnull().sum(),
    'Null_Percentage': (full_df.isnull().sum() / len(full_df)) * 100
})
missing_stats

## 5. Inspection of Rating and Cost Irregularities
Raw ratings contain strings like `'4.1/5'`, `'NEW'`, and `'-'`. Approximate cost contains thousand commas like `'1,200'`.

In [ ]:
print('Distinct Non-Numeric Values in `rate` column:')
print(full_df['rate'].value_counts().head(10))
print('\nSample values in `approx_cost(for two people)` with commas:')
print(full_df[full_df['approx_cost(for two people)'].astype(str).str.contains(',', na=False)]['approx_cost(for two people)'].head(5))

## 6. Duplicate Records & Multi-City Listings
In Zomato, a single restaurant appears repeatedly across multiple search zones (`listed_in(city)`). We evaluate entity-level uniqueness.

In [ ]:
dedup_count = full_df.drop_duplicates(subset=['name', 'location']).shape[0]
print(f'Unique Restaurant Entities (by Name + Location): {dedup_count:,}')
print(f'Redundant Multi-zone Listings: {len(full_df) - dedup_count:,}')

## 7. Geospatial Coordinate Availability Check
Checking whether latitude and longitude columns exist.

In [ ]:
sample_cols = [c.lower() for c in pd.read_csv(RAW_DATA_PATH, nrows=1).columns]
has_lat = 'latitude' in sample_cols or 'lat' in sample_cols
has_lon = 'longitude' in sample_cols or 'lng' in sample_cols
print(f'Latitude Available: {has_lat}')
print(f'Longitude Available: {has_lon}')
print('Finding: GPS coordinates are absent in raw CSV. Following academic integrity rules, ' 
      'no synthetic coordinates will be fabricated. Locality-based heatmaps will serve as advanced visualization.')

## 8. Summary of Initial Observations & Pipeline Strategy
1. **Scale**: 51,717 raw rows representing ~12,500 unique physical dining outlets.
2. **Ratings Cleaning**: Must extract numerator and treat `'NEW'` and `'-'` as `Null` (never 0.0).
3. **Cost Cleaning**: Must strip commas before casting to numeric `DoubleType`.
4. **Distributed Execution**: Next notebook will use Apache Spark Core and DataFrames for distributed processing.